In [1]:
#PANDAS
import pandas as pd

#NUMPY
import numpy as np

#SAVE
import joblib

#SKLEARN
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

#LIGHTGBM
import lightgbm as lgb

#WARNINGS
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ==================== CONFIG ====================
START_DATE = '2016-01-01'
HOLDOUT_DAYS = 16

In [3]:
# ==================== LOAD DATA ====================
print("Loading data...")
train = pd.read_csv('train.csv', parse_dates=['date'])
test = pd.read_csv('test.csv', parse_dates=['date'])
stores = pd.read_csv('stores.csv')
oil = pd.read_csv('oil.csv', parse_dates=['date'])
holidays = pd.read_csv('holidays_events.csv', parse_dates=['date'])

print(f"Original train shape: {train.shape}")

Loading data...
Original train shape: (3000888, 6)


In [4]:
# ==================== FILTER BY DATE ====================
print(f"\nFiltering data from {START_DATE}...")
train = train[train['date'] >= START_DATE].reset_index(drop=True)

print(f"Filtered train shape: {train.shape}")
print(f"Stores used:   {train['store_nbr'].nunique()}")
print(f"Families used: {train['family'].nunique()}")


Filtering data from 2016-01-01...
Filtered train shape: (1054944, 6)
Stores used:   54
Families used: 33


In [5]:
# ==================== MERGE STORE INFO ====================
train = train.merge(stores, on='store_nbr', how='left')
test = test.merge(stores, on='store_nbr', how='left')

In [6]:
# ==================== MERGE OIL PRICE ====================
oil = oil.sort_values('date').reset_index(drop=True)
oil['dcoilwtico'] = oil['dcoilwtico'].ffill().bfill()
oil = oil.rename(columns={'dcoilwtico': 'oil_price'})

train = train.merge(oil, on='date', how='left')
test = test.merge(oil, on='date', how='left')

In [7]:
# ==================== HOLIDAY FLAG ====================
holidays_national = holidays[(holidays['locale'] == 'National') & (holidays['type'] == 'Holiday')][['date']].drop_duplicates()
holidays_national['is_holiday'] = 1

train = train.merge(holidays_national, on='date', how='left')
test = test.merge(holidays_national, on='date', how='left')
train['is_holiday'] = train['is_holiday'].fillna(0).astype(int)
test['is_holiday'] = test['is_holiday'].fillna(0).astype(int)

In [8]:
# ==================== DATE FEATURES ====================
train['year'] = train['date'].dt.year
train['month'] = train['date'].dt.month
train['day'] = train['date'].dt.day
train['dayofweek'] = train['date'].dt.dayofweek
train['weekofyear'] = train['date'].dt.isocalendar().week.astype(int)
train['is_weekend'] = (train['dayofweek'] >= 5).astype(int)

test['year'] = test['date'].dt.year
test['month'] = test['date'].dt.month
test['day'] = test['date'].dt.day
test['dayofweek'] = test['date'].dt.dayofweek
test['weekofyear'] = test['date'].dt.isocalendar().week.astype(int)
test['is_weekend'] = (test['dayofweek'] >= 5).astype(int)

In [9]:
# ==================== LAG & ROLLING FEATURES ====================
print("\nCreating lag and rolling features (this takes ~5-8 minutes for full data)...")

train = train.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)
grouped = train.groupby(['store_nbr', 'family'])['sales']

for lag in [1, 7, 14, 28]:
    train[f'lag_{lag}'] = grouped.shift(lag)

for window in [7, 14, 28]:
    train[f'rolling_mean_{window}'] = grouped.transform(lambda x: x.shift(1).rolling(window).mean())
    
    train[f'rolling_std_{window}'] = grouped.transform(lambda x: x.shift(1).rolling(window).std())

train = train.dropna().reset_index(drop=True)
print(f"Train shape after lag features: {train.shape}")


Creating lag and rolling features (this takes ~5-8 minutes for full data)...
Train shape after lag features: (718146, 28)


In [10]:
# ==================== SAVE LAST 90 DAYS HISTORY ====================
max_date = train['date'].max()
history_start = max_date - pd.Timedelta(days=89)
history = train[train['date'] >= history_start][['date', 'store_nbr', 'family', 'sales']].copy()

# Keep only the last 60 rows per (store, family) to ensure every combo has enough data
history = history.sort_values('date').groupby(['store_nbr', 'family']).tail(60).reset_index(drop=True)

history.to_csv('history.csv', index=False)
print(f"History saved: {history.shape}")
print(f"Min days per combo: {history.groupby(['store_nbr', 'family']).size().min()}")

History saved: (106920, 4)
Min days per combo: 60


In [11]:
# ==================== TRAIN / VALIDATION SPLIT ====================
split_date = max_date - pd.Timedelta(days=HOLDOUT_DAYS)

train_set = train[train['date'] <= split_date].reset_index(drop=True)
val_set = train[train['date'] > split_date].reset_index(drop=True)

print(f"\nTraining rows:   {train_set.shape}")
print(f"Validation rows: {val_set.shape}")
print(f"Split date:      {split_date.date()}")


Training rows:   (696762, 28)
Validation rows: (21384, 28)
Split date:      2017-07-30


In [12]:
# ==================== ENCODE CATEGORICAL DATA ====================
le_family = LabelEncoder()

train_set['family'] = le_family.fit_transform(train_set['family'])
val_set['family'] = le_family.transform(val_set['family'])

le_type = LabelEncoder()
train_set['type'] = le_type.fit_transform(train_set['type'])
val_set['type'] = le_type.transform(val_set['type'])

In [13]:
# ==================== FEATURE SELECTION ====================
features = ['store_nbr', 'family', 'onpromotion', 'type','cluster', 'oil_price', 'is_holiday','year', 'month', 'day', 'dayofweek', 'weekofyear', 'is_weekend',
'lag_1', 'lag_7', 'lag_14', 'lag_28','rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28','rolling_std_7', 'rolling_std_14', 'rolling_std_28']

x_train = train_set[features]
y_train = train_set['sales']
x_val = val_set[features]
y_val = val_set['sales']



In [14]:
# ==================== LOG TRANSFORM ====================
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

In [15]:
# ==================== MODEL SELECTION ====================
print("\nTraining LightGBM on full data...")
model = lgb.LGBMRegressor(n_estimators=800,learning_rate=0.05,num_leaves=64,max_depth=8,random_state=42,n_jobs=-1,verbose=-1)


Training LightGBM on full data...


In [16]:
# ==================== TRAIN ====================
model.fit(x_train, y_train_log,eval_set=[(x_val, y_val_log)],eval_metric='rmse',callbacks=[lgb.early_stopping(50, verbose=False)])

,num_leaves,64
,max_depth,8
,learning_rate,0.05
,n_estimators,800
,random_state,42
,n_jobs,-1
,verbose,-1
,boosting_type,'gbdt'
,subsample_for_bin,200000
,objective,None
,class_weight,None


In [17]:
# ==================== MODEL EVALUATION ====================
y_pred_log = model.predict(x_val)
y_pred = np.expm1(y_pred_log)
y_pred = np.clip(y_pred, 0, None)

def rmsle(y_true, y_pred):
    return np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))

rmsle_score = rmsle(y_val, y_pred)
rmse_score = np.sqrt(mean_squared_error(y_val, y_pred))
mae_score = mean_absolute_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print("\n" + "=" * 60)
print("MODEL PERFORMANCE ON VALIDATION SET")
print("=" * 60)
print(f"RMSLE:  {rmsle_score:.4f}")
print(f"RMSE:   {rmse_score:.4f}")
print(f"MAE:    {mae_score:.4f}")
print(f"R2:     {r2:.4f}")


MODEL PERFORMANCE ON VALIDATION SET
RMSLE:  0.3867
RMSE:   194.6917
MAE:    56.5727
R2:     0.9732


In [18]:
# ==================== FEATURE IMPORTANCE ====================
print("\n" + "=" * 60)
print("TOP 10 FEATURE IMPORTANCE")
print("=" * 60)
importance_df = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).head(10)
print(importance_df.to_string(index=False))


TOP 10 FEATURE IMPORTANCE
       feature  importance
         lag_1         956
           day         873
        family         806
   onpromotion         774
    weekofyear         616
     oil_price         579
rolling_mean_7         515
     dayofweek         512
     store_nbr         501
        lag_14         475


In [19]:
# ==================== SAVE MODEL ====================
joblib.dump(model, 'sales_model.pkl')
joblib.dump(le_family, 'le_family.pkl')
joblib.dump(le_type, 'le_type.pkl')
joblib.dump(features, 'features.pkl')
joblib.dump({
    'oil_last': float(oil['oil_price'].iloc[-1]),
    'rmsle': rmsle_score,
    'rmse': rmse_score,
    'mae': mae_score,
    'r2': r2
}, 'config.pkl')

print("\n✅ Model saved successfully")
print("Now run: streamlit run app.py")#


✅ Model saved successfully
Now run: streamlit run app.py
